### STEP 2. 실제 스키마·PK·FK를 먼저 검증


In [1]:
import pandas as pd
from pathlib import Path

# 1) 전처리된 파일이 있는 폴더 (노트북이 chapter12 폴더 안에 있으므로 한 단계 위)
data_dir = Path("../data/processed")

# 2) 테이블별로 "꼭 있어야 하는 컬럼" 정의
required_columns = {
    "customers":   ["customer_id", "name", "gender", "age", "city", "signup_date"],
    "products":    ["product_id", "product_name", "category", "price"],
    "orders":      ["order_id", "customer_id", "order_date", "payment_method", "order_status"],
    "order_items": ["order_item_id", "order_id", "product_id", "quantity", "unit_price", "line_total"],
}

# 3) processed 폴더의 CSV 읽기 (파일이 없으면 raw로 대체하지 않고 바로 멈춤)
datasets = {}
for name in required_columns:
    path = data_dir / f"{name}_clean.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} 없음 → python scripts/preprocess_data.py 먼저 실행")
    datasets[name] = pd.read_csv(path)

# 4) 필수 컬럼이 실제로 있는지 하나씩 확인
rows = []
for name, cols in required_columns.items():
    actual = list(datasets[name].columns)
    missing = [c for c in cols if c not in actual]   # 있어야 하는데 없는 컬럼
    rows.append({
        "table": name,
        "required": len(cols),
        "missing_columns": ", ".join(missing) if missing else "-",
        "result": "FAIL" if missing else "PASS",
    })

required_column_check = pd.DataFrame(rows)
display(required_column_check)

# 5) 하나라도 FAIL이면 다음 단계로 진행하지 않음
assert (required_column_check["result"] == "PASS").all(), "필수 컬럼 누락 → 데이터 준비로 돌아가세요"
print("✅ 모든 테이블의 필수 컬럼이 존재합니다.")

,table,required,missing_columns,result
0,customers,6,-,PASS
1,products,4,-,PASS
2,orders,5,-,PASS
3,order_items,6,-,PASS


✅ 모든 테이블의 필수 컬럼이 존재합니다.
